In [48]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/dhruvtaneja31/rice-grain-classification/classification_224 new/test/1509_sella/1 copy 16.jpg
/kaggle/input/datasets/dhruvtaneja31/rice-grain-classification/classification_224 new/test/1509_sella/3 copy 8.jpg
/kaggle/input/datasets/dhruvtaneja31/rice-grain-classification/classification_224 new/test/1509_sella/1 copy 6.jpg
/kaggle/input/datasets/dhruvtaneja31/rice-grain-classification/classification_224 new/test/1509_sella/1 copy 5.jpg
/kaggle/input/datasets/dhruvtaneja31/rice-grain-classification/classification_224 new/test/1509_sella/3 copy 5.jpg
/kaggle/input/datasets/dhruvtaneja31/rice-grain-classification/classification_224 new/test/1509_sella/1 copy 17.jpg
/kaggle/input/datasets/dhruvtaneja31/rice-grain-classification/classification_224 new/test/1509_sella/1 copy 14.jpg
/kaggle/input/datasets/dhruvtaneja31/rice-grain-classification/classification_224 new/test/1509_sella/3 copy 10.jpg
/kaggle/input/datasets/dhruvtaneja31/rice-grain-classification/classificatio

In [49]:
from tensorflow.keras.applications import ResNet50
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input

In [50]:
train_path = "/kaggle/input/datasets/dhruvtaneja31/rice-grain-classification/classification_224 new/train"
val_path   = "/kaggle/input/datasets/dhruvtaneja31/rice-grain-classification/classification_224 new/train"
test_path  = "/kaggle/input/datasets/dhruvtaneja31/rice-grain-classification/classification_224 new/test"

In [51]:
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.1,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_generator = datagen.flow_from_directory(
    val_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=True
)

Found 512 images belonging to 4 classes.
Found 128 images belonging to 4 classes.


In [52]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

In [53]:
for layer in base_model.layers:
    layer.trainable = False

In [54]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(4, activation='softmax')
])

In [55]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3,
    min_lr=1e-6
)

from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = train_generator.classes

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(classes),
    y=classes
)

class_weights = dict(enumerate(class_weights))

In [56]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [58]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weights
)

Epoch 1/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 11s 709ms/step - accuracy: 0.3192 - loss: 1.6146 - val_accuracy: 0.2734 - val_loss: 1.4095 - learning_rate: 1.0000e-04
Epoch 2/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 431ms/step - accuracy: 0.3288 - loss: 1.4345 - val_accuracy: 0.3047 - val_loss: 1.3644 - learning_rate: 1.0000e-04
Epoch 3/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 422ms/step - accuracy: 0.3681 - loss: 1.3916 - val_accuracy: 0.3203 - val_loss: 1.2954 - learning_rate: 1.0000e-04
Epoch 4/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 427ms/step - accuracy: 0.4458 - loss: 1.2327 - val_accuracy: 0.4219 - val_loss: 1.2809 - learning_rate: 1.0000e-04
Epoch 5/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 422ms/step - accuracy: 0.4882 - loss: 1.1641 - val_accuracy: 0.3594 - val_loss: 1.2995 - learning_rate: 1.0000e-04
Epoch 6/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 425ms/step - accuracy: 0.4986 - loss: 1.1288 - val_accuracy: 0.4688 - val_loss: 1.1665 - learning_rate: 1.0000e-04
Epoch 7/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 420ms/step - accuracy:

In [ ]:
for layer in base_model.layers[-20:]:
    layer.trainable = True

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_finetune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

In [ ]:
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

test_generator = test_datagen.flow_from_directory(
    test_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

test_loss, test_accuracy = model.evaluate(test_generator)

print("Test Accuracy:", test_accuracy)
print("Test Loss:", test_loss)